# 01. Tổng quan bộ dữ liệu

Kiểm tra bộ dữ liệu ứng viên trước khi nhóm quyết định có dùng cho phân loại CWE hay không.

Notebook này chỉ khảo sát. Không xóa dòng, không chọn ngưỡng CWE, không lấy CWE đầu tiên làm nhãn, không chốt chiến lược chia dữ liệu.

Mọi số liệu phải tính từ file đã tải. Không điền số từ bài báo. Logic dùng lại được thì đưa vào `src/data/`, không viết quy trình nghiên cứu ở đây.


## 0. Thiết lập

Chạy Jupyter từ thư mục gốc repository sau khi `pip install -r requirements.txt`.

Bộ dữ liệu ứng viên hiện tại là DiverseVul. File cấu hình: `configs/data/diversevul.yaml`. Đặt dữ liệu vào `data/raw/diversevul/` và ghi phiên bản trong cấu hình. Không commit dữ liệu thô.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ROOT = ROOT.resolve()
if not (ROOT / "src").is_dir():
    raise RuntimeError("Không thể xác định thư mục gốc repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.config import load_config
from src.utils.paths import report_dir, resolve_from_root

config = load_config(ROOT / "configs" / "data" / "diversevul.yaml")
dataset = config["dataset"]
raw_dir = resolve_from_root(dataset["local_raw_dir"])
table_dir = report_dir("dataset", "tables")
figure_dir = report_dir("dataset", "figures")

print("dataset:", dataset["name"])
print("status:", dataset["status"])
print("version:", dataset["version"])
print("raw_dir:", raw_dir)
print("raw_dir_exists:", raw_dir.exists())
print("files:", sorted(path.name for path in raw_dir.glob("*")) if raw_dir.exists() else [])
print("Bước tiếp theo: load file bằng src.data.loader.load_dataset. Loader này hiện vẫn là khung triển khai.")


dataset: diversevul
status: candidate
version: None
raw_dir: D:\PROJECT\software-bug-detection-using-graphs\data\raw\diversevul
raw_dir_exists: False
files: []
Bước tiếp theo: load file bằng src.data.loader.load_dataset. Loader này hiện vẫn là khung triển khai.


## A. Tổng quan bộ dữ liệu

Tính từ dữ liệu thật:

- Bộ dữ liệu có bao nhiêu bản ghi?
- Bao nhiêu hàm có lỗ hổng?
- Bao nhiêu hàm không có lỗ hổng?
- Tỷ lệ hàm có lỗ hổng là bao nhiêu?
- Có những cột hoặc trường nào?
- Số giá trị thiếu của từng trường?
- Mã nguồn có bị thiếu không?
- Siêu dữ liệu dự án có bị thiếu không?
- Siêu dữ liệu commit có bị thiếu không?
- Siêu dữ liệu CWE có bị thiếu không?

Ghi cấu trúc thật vào cấu hình (`code_field`, `label_field`, `cwe_field`, `project_field`, `commit_field`, `hash_field`, `timestamp_field`) sau khi đã kiểm tra. Không đoán tên cột.


In [2]:
# TODO: triển khai src.data.loader.load_dataset, sau đó thay điểm dừng này bằng phần tóm tắt frame.
# Không in toàn bộ frame.
if not raw_dir.exists() or not any(raw_dir.iterdir()):
    print("ĐÃ DỪNG: không có file dataset trong", raw_dir)
    print("Tải DiverseVul thủ công, đặt dataset.version, sau đó chạy lại từ đầu.")
else:
    raise NotImplementedError(
        "Đã có file. Hãy triển khai load_dataset và tính số record, "
        "class balance, danh sách cột và missing values tại đây."
    )


ĐÃ DỪNG: không có file dataset trong D:\PROJECT\software-bug-detection-using-graphs\data\raw\diversevul
Tải DiverseVul thủ công, đặt dataset.version, sau đó chạy lại từ đầu.


## B. Phân tích CWE

Chỉ phân tích CWE có thật trong dữ liệu. Hàm dự kiến: `src.data.cwe.cwe_distribution` và `src.data.cwe.threshold_summary`.

Cần tính:

- số CWE duy nhất
- số hàm có lỗ hổng và có CWE
- số hàm có lỗ hổng nhưng không có CWE
- số mẫu của từng CWE
- tỷ lệ phần trăm của từng CWE
- 10 CWE phổ biến nhất
- 20 CWE phổ biến nhất
- dạng phân bố đuôi dài

Xuất `reports/dataset/tables/cwe_distribution.csv`:

| CWE | sample_count | percentage | project_count |
| --- | --- | --- | --- |

Với từng ngưỡng `>= 20`, `>= 50`, `>= 100`, `>= 200`, báo cáo số CWE còn lại, số hàm có lỗ hổng còn lại và phần trăm dữ liệu giữ lại. Không chọn ngưỡng.


In [3]:
print("TODO: gọi src.data.cwe.cwe_distribution(records)")
print("TODO: gọi src.data.cwe.threshold_summary(distribution)")
print("Ghi reports/dataset/tables/cwe_distribution.csv")
print("Các threshold cần báo cáo mà không tự chọn một ngưỡng: 20, 50, 100, 200")


TODO: gọi src.data.cwe.cwe_distribution(records)
TODO: gọi src.data.cwe.threshold_summary(distribution)
Ghi reports/dataset/tables/cwe_distribution.csv
Các threshold cần báo cáo mà không tự chọn một ngưỡng: 20, 50, 100, 200


## C. Phân tích multi-CWE

Hàm dự kiến: `src.data.cwe.parse_cwe_labels` và `src.data.cwe.multi_cwe_summary`.

Báo cáo:

- số hàm có đúng 1 CWE
- số hàm có từ 2 CWE trở lên
- tỷ lệ multi-CWE
- phân bố số CWE trên một mẫu
- vài ví dụ multi-CWE

Không lấy CWE đầu tiên làm nhãn chuẩn.


In [4]:
print("TODO: gọi src.data.cwe.multi_cwe_summary(records)")
print("Chỉ hiển thị một số ví dụ multi-CWE, không hiển thị toàn bộ dataset.")


TODO: gọi src.data.cwe.multi_cwe_summary(records)
Chỉ hiển thị một số ví dụ multi-CWE, không hiển thị toàn bộ dataset.


## D. Phân tích dự án

Xuất:

- `reports/dataset/tables/project_distribution.csv` với `project`, `sample_count`, `vulnerable_count`, `cwe_count`
- `reports/dataset/tables/cwe_project_distribution.csv` với `CWE`, `sample_count`, `project_count`

Cần trả lời:

- số dự án
- số mẫu trên mỗi dự án
- số mẫu có lỗ hổng trên mỗi dự án
- số CWE trên mỗi dự án
- số dự án trên mỗi CWE
- chia ngẫu nhiên có khả thi không
- chia theo dự án có khả thi không
- siêu dữ liệu thời gian có đủ để chia theo thời gian không

Một CWE có nhiều mẫu nhưng chỉ nằm trong một dự án là rủi ro cho cách chia theo dự án. Ghi các trường hợp đó. Chưa chốt cách chia dữ liệu.


In [5]:
print("TODO: tạo bảng project và bảng CWE theo project.")
print("Đánh dấu các CWE có project_count bằng 1.")
print("Ghi nhận field timestamp có tồn tại hay không và có bao nhiêu giá trị bị thiếu.")


TODO: tạo bảng project và bảng CWE theo project.
Đánh dấu các CWE có project_count bằng 1.
Ghi nhận field timestamp có tồn tại hay không và có bao nhiêu giá trị bị thiếu.


## E. Phân tích dữ liệu trùng lặp

Chỉ báo cáo, không xóa. Hàm dự kiến nằm trong `src.data.deduplication`.

Kiểm tra:

- mã nguồn trùng khớp chính xác
- trùng mã băm nếu bộ dữ liệu có cột mã băm
- cùng mã nguồn nhưng khác nhãn
- cùng một hàm xuất hiện trong nhiều bản ghi
- trùng sau khi chuẩn hóa ký tự xuống dòng và khoảng trắng
- trùng sau khi bỏ chú thích, chỉ khi cách bỏ chú thích an toàn với C/C++


In [6]:
print("TODO: exact_source_duplicates, hash_duplicates, conflicting_label_duplicates")
print("TODO: normalized_source_duplicates via src.data.preprocessing")
print("Giữ nguyên mọi dòng dữ liệu gốc.")


TODO: exact_source_duplicates, hash_duplicates, conflicting_label_duplicates
TODO: normalized_source_duplicates via src.data.preprocessing
Giữ nguyên mọi dòng dữ liệu gốc.


## F. Chất lượng mã nguồn

Kiểm tra hàm rỗng, hàm cực ngắn, hàm cực dài, phân bố số dòng, phân bố độ dài ký tự hoặc token và mẫu có vẻ sai cấu trúc nếu phát hiện được.

Lấy ngẫu nhiên một ít hàm có lỗ hổng để xem thủ công. Không in toàn bộ bộ dữ liệu.


In [7]:
print("TODO: tóm tắt số dòng và độ dài theo ký tự.")
print("TODO: in một sample nhỏ gồm các vulnerable function để kiểm tra thủ công.")


TODO: tóm tắt số dòng và độ dài theo ký tự.
TODO: in một sample nhỏ gồm các vulnerable function để kiểm tra thủ công.


## G. Mức sẵn sàng để tạo đồ thị

Chưa sinh CPG cho toàn bộ dữ liệu. CPG và Joern vẫn là phương án ứng viên.

Chỉ cần kết luận mã nguồn có đủ để bước trích đồ thị chạy thử hay không. Nếu thuận tiện, ghi một danh mục khoảng 20 đến 50 hàm (ID và đường dẫn) để giai đoạn sau gọi Joern. Không chạy Joern trong notebook này.


In [8]:
print("TODO: đếm sample có source code bị thiếu hoặc rỗng.")
print("TODO: có thể ghi manifest gồm 20-50 function vào data/interim và lưu lại đường dẫn.")
print("Không export CPG cho toàn bộ dataset.")


TODO: đếm sample có source code bị thiếu hoặc rỗng.
TODO: có thể ghi manifest gồm 20-50 function vào data/interim và lưu lại đường dẫn.
Không export CPG cho toàn bộ dataset.


## Kết luận cần ghi vào báo cáo

Sau khi các mục trên có số, cập nhật `reports/dataset/dataset_eda.md`. Mỗi kết luận phải dẫn từ bảng vừa tính:

1. DiverseVul có phù hợp với phân loại CWE hay không.
2. Có bao nhiêu CWE thực sự sử dụng được, kèm ngưỡng đang được xem xét chứ chưa chốt.
3. Mất cân bằng lớp nghiêm trọng tới mức nào.
4. Multi-CWE có phải vấn đề đáng kể không.
5. Duplicate có nghiêm trọng không.
6. Chia dữ liệu theo dự án có khả thi không.
7. Mã nguồn có đủ điều kiện để thử trích xuất đồ thị không.
8. Nhóm cần quyết định gì tiếp theo.

Không tuyên bố DiverseVul là bộ dữ liệu cuối cùng nếu bằng chứng chưa đủ. Nếu số khác paper, ghi số của file đang dùng và phiên bản dữ liệu.


In [9]:
print("Template báo cáo:", report_dir("dataset", "dataset_eda.md"))
print("Các bảng:", table_dir)
print("Các biểu đồ:", figure_dir)
print("Các bảng dự kiến:")
print("- cwe_distribution.csv")
print("- project_distribution.csv")
print("- cwe_project_distribution.csv")


Template báo cáo: D:\PROJECT\software-bug-detection-using-graphs\reports\dataset\dataset_eda.md
Các bảng: D:\PROJECT\software-bug-detection-using-graphs\reports\dataset\tables
Các biểu đồ: D:\PROJECT\software-bug-detection-using-graphs\reports\dataset\figures
Các bảng dự kiến:
- cwe_distribution.csv
- project_distribution.csv
- cwe_project_distribution.csv
